In [129]:
import os
from dotenv import load_dotenv

from llama_index.llms.google_genai import GoogleGenAI
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import (
    AgentInput,
    AgentOutput,
    ToolCall,
    ToolCallResult,
    AgentStream,
    FunctionAgent, 
    AgentWorkflow
)

from typing import Union, Dict, Any
import json
import unicodedata

In [130]:
class AnalysisResult(BaseModel):
    original_langauge: str = Field(description="The langauge of the original text")
    language_style: str = Field(description="The language style of the original text")
    main_message : str = Field(description="The main message of the original text")
    historical_background : str = Field(description="The historical background of the original text")
    culture_behind: str = Field(description="The cultural information of the original text")

class TranslatedResult(BaseModel):
    translated_text: str = Field(description="The translated text")
    original_langauge: str = Field(description="The langauge of the original text")
    target_langauge: str = Field(description="The langauge of the translated text")

In [131]:
load_dotenv()
project_name = os.getenv('PROJECT_NAME')

model = "gemini-2.0-flash-001"

In [132]:
async def translate_text(ctx: Context, translated_text: str, source_language: str, target_language) -> str:
    """Useful for translating input text from source language to target language."""
    current_state = await ctx.get("state")

    translated_key = f"{source_language}_{target_language}"
    if "translation" not in current_state:
        current_state["translation"] = {}
    current_state["translated_text"][translated_key] = translated_text
    await ctx.set("state", current_state)
    return "Translation Done."

async def get_input_text(ctx: Context, file_name: str) -> str:
    """Useful for getting input text from user."""
    with open(f"{file_name}", "r", encoding="utf8") as f:
        japanese_story = f.read()
        japanese_story = unicodedata.normalize('NFKC', japanese_story)
    current_state = await ctx.get("state")
    current_state["input_text"] = japanese_story
    await ctx.set("state", current_state)
    return "Input text received."


In [ ]:
system_prompt = """You are a professional translator. 
You only know Japanese, Chinese and English. 
You can translate Japanese into either Chinese or English. 
You can also translate Chinese into English, and English into Chinese.
All the translations should keep the original meaning.
All the translations only contain characters of the source and target languages.
For Chinese, display in Traditional Chinese.

You are now working on Japanese text which is a translation of an Ainu chant, sung by Ainu gods telling their story. 
"""

llm = GoogleGenAI(
    model=model,
    vertexai_config={"project": project_name, "location": "us-central1"},
    # you should set the context window to the max input tokens for the model
    max_tokens=8192,
    #system_prompt=system_prompt,
    temperature=0.0,
    top_p=0.0,
)

In [134]:
get_input_text_agent = FunctionAgent(
    name="InputAgent",
    description="Useful for getting input text.",
    system_prompt=(
        "You are the InputAgent that reads the input text from user. "
        "The user will provide you with the file name of the input text file. "
    ),
    llm=llm,
    tools=[get_input_text],
)

In [135]:
song_no = 1
with open(f"Chiri_Japanese_Translation/story_translation_{song_no}.txt", "r", encoding="utf8") as f:
    japanese_story = f.read()
    japanese_story = unicodedata.normalize('NFKC', japanese_story)

In [136]:
agent_workflow = AgentWorkflow(
    agents=[get_input_text_agent,translate_agent],
    root_agent=get_input_text_agent.name,
    initial_state={
        "input_text": {},
    },
)

In [137]:
handler = agent_workflow.run(
    input=(
       """The following file "Chiri_Japanese_Translation/story_translation_1.txt" contains the input text. Open the file and read the text."""
    )
)

In [138]:
current_agent = None
current_tool_calls = ""
async for event in handler.stream_events():
    if (
        hasattr(event, "current_agent_name")
        and event.current_agent_name != current_agent
    ):
        current_agent = event.current_agent_name
        print(f"\n{'='*50}")
        print(f"🤖 Agent: {current_agent}")
        print(f"{'='*50}\n")

    elif isinstance(event, AgentOutput):
        if event.response.content:
            print("📤 Output:", event.response.content)
        if event.tool_calls:
            print(
                "🛠️  Planning to use tools:",
                [call.tool_name for call in event.tool_calls],
            )
    elif isinstance(event, ToolCallResult):
        print(f"🔧 Tool Result ({event.tool_name}):")
        print(f"  Arguments: {event.tool_kwargs}")
        print(f"  Output: {event.tool_output}")
    elif isinstance(event, ToolCall):
        print(f"🔨 Calling Tool: {event.tool_name}")
        print(f"  With arguments: {event.tool_kwargs}")

In [140]:
current_agent

    